# Manual Gradient Descent for Initial Condition Optimization

This notebook optimizes initial conditions (x_0) using manual gradient descent with visualization of RMSE and predictions at each step.

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import xarray as xr
import numpy as np
import os
import sys
from pathlib import Path
from datetime import datetime
from typing import Tuple, Dict
import gc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import clear_output, display
import pandas as pd

# Add paths
sys.path.append(str(Path.cwd().parent / "glonet_daily_forecast_local"))
sys.path.append(str(Path.cwd().parent / "src/glonet"))
sys.path.append(str(Path.cwd() / "src"))
 
from utility import get_normalizer1, get_normalizer2, get_normalizer3
from utility import get_denormalizer1, get_denormalizer2, get_denormalizer3
from modelp2 import Glonet
from optimIC_GD_glonetLit import GlonetGradientCheckpointing

# Set matplotlib style
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ All imports successful")

## 2. Configuration

In [ ]:
# Configuration parameters
CONFIG = {
    'data_path': "/Odyssey/public/glonet/glorys12_1993-01-01_to_1993-06-30_init_states/combined_input.nc", 
    'model_location': "/Odyssey/public/glonet/TrainedWeights",
    'learning_rate': 1e0,
    'num_iterations': 500,
    'sample_idx': 0,
    'sequence_length': 2,
    'forecast_horizon': 1,
    'device': "cuda:0" if torch.cuda.is_available() else "cpu",
    'output_dir': f"outputs/man_optimIC_GD/notebook_optim_smoothsig2{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    'plot_frequency': 10,  # Plot every N iterations
    'gradient_filter': 'none',  # Options: 'none', 'weight', 'delta', 'pooling', 'pooling+weight'
    'pooling_kernel': 4,  # Kernel size for average pooling (must divide evenly into spatial dimensions)
    # Spatial smoothing parameters
    'spatial_smoothing': True,  # Whether to apply spatial smoothing to predictions and targets
    'smooth_kernel_size': 5,  # Gaussian kernel size (must be odd)
    'smooth_sigma': 2,  # Gaussian sigma (larger = more smoothing)
}

# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 3. Load Models and Normalizers

In [ ]:
print("Loading normalizers and denormalizers...")
normalizer1 = get_normalizer1(CONFIG['model_location'])
normalizer2 = get_normalizer2(CONFIG['model_location'])
normalizer3 = get_normalizer3(CONFIG['model_location'])

denormalizer1 = get_denormalizer1(CONFIG['model_location'])
denormalizer2 = get_denormalizer2(CONFIG['model_location'])
denormalizer3 = get_denormalizer3(CONFIG['model_location'])

print("✓ Normalizers loaded")

In [ ]:
def load_checkpoint_model(checkpoint_path: str, shape_in: Tuple, device: str):
    """Load a checkpoint-based model with gradient checkpointing."""
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model = GlonetGradientCheckpointing(shape_in=shape_in)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.train()  # Enable gradient checkpointing
    for param in model.parameters():
        param.requires_grad = False
    return model

print("Loading pretrained models...")
model1 = load_checkpoint_model(
    f"{CONFIG['model_location']}/glonet_part1.pth",
    shape_in=(2, 5, 672, 1440),
    device=CONFIG['device']
)
model2 = load_checkpoint_model(
    f"{CONFIG['model_location']}/glonet_part2.pth",
    shape_in=(2, 40, 672, 1440),
    device=CONFIG['device']
)
model3 = load_checkpoint_model(
    f"{CONFIG['model_location']}/glonet_part3.pth",
    shape_in=(2, 40, 672, 1440),
    device=CONFIG['device']
)

loss_fn = nn.MSELoss()

print("✓ Models loaded successfully")

## 4. Load and Prepare Data

In [ ]:
print(f"Loading dataset from {CONFIG['data_path']}...")
dataset = xr.open_dataset(CONFIG['data_path'])

# Extract input sequence and targets for each forecast day
input_sequence = dataset.isel(
    time=slice(CONFIG['sample_idx'], CONFIG['sample_idx'] + CONFIG['sequence_length'])
)

# Load targets for each day in forecast horizon
targets = []
target_times = []
for day in range(CONFIG['forecast_horizon']):
    target_time_idx = CONFIG['sample_idx'] + CONFIG['sequence_length'] + day
    target = dataset.isel(time=target_time_idx)
    targets.append(target)
    target_times.append(target['time'].values)

# Store coordinates
coords = {
    'time': input_sequence['time'].values,
    'target_times': target_times,
    'lat': input_sequence['lat'].values,
    'lon': input_sequence['lon'].values
}

print(f"Input sequence shape: {input_sequence['data'].shape}")
print(f"Number of targets: {len(targets)} (forecast horizon={CONFIG['forecast_horizon']})")
for i, target in enumerate(targets):
    print(f"  Target {i+1} shape: {target['data'].shape}, time: {target_times[i]}")

In [ ]:
# Create ocean masks
sample_data = dataset.isel(time=CONFIG['sample_idx'])
data_values = sample_data['data'].values
land_mask = np.isnan(data_values).astype(np.float32)

ocean_mask_1 = torch.from_numpy((1.0 - land_mask[0:5, :, :]).copy()).float().to(CONFIG['device'])
ocean_mask_2 = torch.from_numpy((1.0 - land_mask[5:45, :, :]).copy()).float().to(CONFIG['device'])
ocean_mask_3 = torch.from_numpy((1.0 - land_mask[45:85, :, :]).copy()).float().to(CONFIG['device'])
ocean_mask_all = torch.cat([ocean_mask_1, ocean_mask_2, ocean_mask_3], dim=0)

print(f"Ocean mask 1: {ocean_mask_1.shape}")
print(f"Ocean mask 2: {ocean_mask_2.shape}")
print(f"Ocean mask 3: {ocean_mask_3.shape}")

In [ ]:
# Prepare initial conditions and targets
input_data = input_sequence['data'].values
input_data = np.nan_to_num(input_data, nan=0.0)

x0 = torch.from_numpy(input_data.copy()).float().unsqueeze(0).to(CONFIG['device'])
x0_ref = x0.clone()

# Prepare target tensors for each forecast day
target_tensors = []
for target in targets:
    target_data = target['data'].values
    target_data = np.nan_to_num(target_data, nan=0.0)
    target_tensor = torch.from_numpy(target_data.copy()).float().unsqueeze(0).to(CONFIG['device'])
    target_tensors.append(target_tensor)

x0.requires_grad = True
x0_ref.requires_grad = False

print(f"Initial condition shape: {x0.shape}")
print(f"Number of target tensors: {len(target_tensors)}")
for i, target_tensor in enumerate(target_tensors):
    print(f"  Target {i+1} shape: {target_tensor.shape}")
print("✓ Data preparation complete")

## 5. Define Functions

In [ ]:
def forward(x):
    """Forward pass through the models. Returns predictions at each forecast step."""
    
    predictions = []  # Store predictions at each forecast step
    
    with torch.enable_grad():
        x1 = x[:, :, 0:5, :, :] 
        x1_in_std = normalizer1(x1) * ocean_mask_1.unsqueeze(0).unsqueeze(0)
        x2 = x[:, :, 5:45, :, :]
        x2_in_std = normalizer2(x2)  * ocean_mask_2.unsqueeze(0).unsqueeze(0)
        x3 = x[:, :, 45:85, :, :] 
        x3_in_std = normalizer3(x3) * ocean_mask_3.unsqueeze(0).unsqueeze(0)
        
        y_hat1 = model1(x1_in_std)
        y_hat2 = model2(x2_in_std)
        y_hat3 = model3(x3_in_std)
        
        # Denormalize and store first prediction
        y_hat1_denorm = denormalizer1(y_hat1[:, -1, :, :, :])
        y_hat2_denorm = denormalizer2(y_hat2[:, -1, :, :, :])
        y_hat3_denorm = denormalizer3(y_hat3[:, -1, :, :, :])
        y_hat_step = torch.cat([y_hat1_denorm, y_hat2_denorm, y_hat3_denorm], dim=1)
        predictions.append(y_hat_step)
            
        # Auto regressive forecasting for forecast_horizon steps
        # Following forecast.py: output goes directly back as input
        if CONFIG['forecast_horizon'] > 1 :
            for i in range(CONFIG['forecast_horizon'] - 1) :
                y_hat1 = model1(y_hat1)
                y_hat2 = model2(y_hat2)
                y_hat3 = model3(y_hat3)
                
                # Denormalize and store prediction at each step
                y_hat1_denorm = denormalizer1(y_hat1[:, -1, :, :, :])
                y_hat2_denorm = denormalizer2(y_hat2[:, -1, :, :, :])
                y_hat3_denorm = denormalizer3(y_hat3[:, -1, :, :, :])
                y_hat_step = torch.cat([y_hat1_denorm, y_hat2_denorm, y_hat3_denorm], dim=1)
                predictions.append(y_hat_step)
    
    x0_in_std = torch.cat([x1_in_std, x2_in_std, x3_in_std], dim=2)
    
    return x0_in_std, predictions

def compute_loss(predictions, targets, apply_smoothing_filter=False, smooth_kernel_size=5, smooth_sigma=1.0) :
    """Compute loss as sum of normalized squared errors over all pixels for each forecast day.
    
    Args:
        predictions: List of prediction tensors, one for each forecast day [1, 85, H, W]
        targets: List of target tensors, one for each forecast day [1, 85, H, W]
        apply_smoothing_filter: Whether to apply spatial smoothing before computing loss
        smooth_kernel_size: Kernel size for Gaussian smoothing (if enabled)
        smooth_sigma: Sigma for Gaussian smoothing (if enabled)
    
    Returns:
        mse: Mean squared error summed over all forecast days [1, 85]
        nmse: Normalized mean squared error summed over all forecast days [1, 85]
    """
    
    # Initialize accumulators
    total_mse = None
    total_nmse = None
    
    # Compute loss for each forecast day and sum
    for day_idx, (y_hat, target_tensor) in enumerate(zip(predictions, targets)):
        # Apply spatial smoothing if enabled
        if apply_smoothing_filter:
            y_hat_smooth = apply_spatial_smoothing(
                y_hat, 
                kernel_size=smooth_kernel_size, 
                sigma=smooth_sigma, 
                ocean_mask=ocean_mask_all
            )
            target_smooth = apply_spatial_smoothing(
                target_tensor, 
                kernel_size=smooth_kernel_size, 
                sigma=smooth_sigma, 
                ocean_mask=ocean_mask_all
            )
        else:
            y_hat_smooth = y_hat
            target_smooth = target_tensor
        
        se = ((target_smooth - y_hat_smooth) ** 2)                      # [1, 85, 672, 1440]
        se_masked = se * ocean_mask_all.unsqueeze(0)                    # [1, 85, 672, 1440]
        n_ocean_points = ocean_mask_all.unsqueeze(0).sum(dim=(2, 3))    # [1, 85]
        mse = se_masked.sum(dim=(2, 3)) / (n_ocean_points + 1e-10)      # [1, 85]
        
        var = target_smooth.var(dim=(2, 3))     # [1, 85]
        nmse = mse / (var + 1e-10)              # [1, 85]
        
        # Accumulate
        if total_mse is None:
            total_mse = mse
            total_nmse = nmse
        else:
            total_mse = total_mse + mse
            total_nmse = total_nmse + nmse
    
    return total_mse, total_nmse

print("✓ Forward and loss functions defined")

In [ ]:
def apply_gradient_filter(gradient, x0, filter_type='none'):
    """Apply different filtering strategies to gradients.
    
    Args:
        gradient: Raw gradient tensor [1, T, 85, H, W]
        x0: Current initial condition [1, T, 85, H, W]
        filter_type: Type of filter to apply
            - 'none': No filtering (return raw gradient)
            - 'weight': Weight by variable variance
            - 'delta': Adaptive step size based on gradient magnitude
            - 'pooling': Average pooling + upsampling for global optimization
            - 'pooling+weight': Combination of pooling and variance weighting
    
    Returns:
        filtered_gradient: Filtered gradient tensor [1, T, 85, H, W]
    """
    filtered_grad = gradient.clone()
    
    if filter_type == 'none':
        return filtered_grad
    
    elif filter_type == 'weight':
        # Weight gradients by variable variance (current approach)
        with torch.no_grad():
            var_0 = x0[:, 0, 0, :, :].var().item() + 1e-10
            var_1 = x0[:, 0, 1, :, :].var().item() + 1e-10
            var_2 = x0[:, 0, 2, :, :].var().item() + 1e-10
            var_3 = x0[:, 0, 3, :, :].var().item() + 1e-10
            var_4 = x0[:, 0, 4, :, :].var().item() + 1e-10
            
            weight = torch.ones_like(filtered_grad)
            weight[:, :, 0, :, :] *= var_0
            weight[:, :, 1, :, :] *= var_1
            weight[:, :, 2, :, :] *= var_2
            weight[:, :, 3, :, :] *= var_3
            weight[:, :, 4, :, :] *= var_4
            
            filtered_grad = filtered_grad * weight
    
    elif filter_type == 'delta':
        # Adaptive filtering: normalize by gradient magnitude per channel
        with torch.no_grad():
            for t_idx in range(filtered_grad.shape[1]):
                for ch_idx in range(filtered_grad.shape[2]):
                    grad_ch = filtered_grad[0, t_idx, ch_idx]
                    ocean_points = grad_ch[ocean_mask_all[ch_idx] > 0]
                    if len(ocean_points) > 0:
                        grad_norm = torch.norm(ocean_points)
                        if grad_norm > 1e-10:
                            filtered_grad[0, t_idx, ch_idx] = grad_ch / (grad_norm + 1e-10)
    
    elif filter_type == 'pooling':
        # Average pooling + upsampling for global optimization
        kernel_size = CONFIG['pooling_kernel']
        
        with torch.no_grad():
            # Apply pooling and upsampling per channel
            for t_idx in range(filtered_grad.shape[1]):
                for ch_idx in range(filtered_grad.shape[2]):
                    grad_ch = filtered_grad[0, t_idx, ch_idx].unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
                    
                    # Average pooling
                    pooled = nn.functional.avg_pool2d(grad_ch, kernel_size=kernel_size, stride=kernel_size)
                    
                    # Upsample back to original size
                    upsampled = nn.functional.interpolate(
                        pooled, 
                        size=(grad_ch.shape[2], grad_ch.shape[3]), 
                        mode='bilinear', 
                        align_corners=False
                    )
                    
                    filtered_grad[0, t_idx, ch_idx] = upsampled.squeeze(0).squeeze(0)
                    
                    # Re-apply ocean mask
                    filtered_grad[0, t_idx, ch_idx] *= ocean_mask_all[ch_idx]
    
    elif filter_type == 'pooling+weight':
        # Combination: first apply pooling, then variance weighting
        kernel_size = CONFIG['pooling_kernel']
        
        with torch.no_grad():
            # Step 1: Pooling
            for t_idx in range(filtered_grad.shape[1]):
                for ch_idx in range(filtered_grad.shape[2]):
                    grad_ch = filtered_grad[0, t_idx, ch_idx].unsqueeze(0).unsqueeze(0)
                    pooled = nn.functional.avg_pool2d(grad_ch, kernel_size=kernel_size, stride=kernel_size)
                    upsampled = nn.functional.interpolate(
                        pooled, 
                        size=(grad_ch.shape[2], grad_ch.shape[3]), 
                        mode='bilinear', 
                        align_corners=False
                    )
                    filtered_grad[0, t_idx, ch_idx] = upsampled.squeeze(0).squeeze(0)
                    filtered_grad[0, t_idx, ch_idx] *= ocean_mask_all[ch_idx]
            
            # Step 2: Variance weighting
            var_0 = x0[:, 0, 0, :, :].var().item() + 1e-10
            var_1 = x0[:, 0, 1, :, :].var().item() + 1e-10
            var_2 = x0[:, 0, 2, :, :].var().item() + 1e-10
            var_3 = x0[:, 0, 3, :, :].var().item() + 1e-10
            var_4 = x0[:, 0, 4, :, :].var().item() + 1e-10
            
            weight = torch.ones_like(filtered_grad)
            weight[:, :, 0, :, :] *= var_0
            weight[:, :, 1, :, :] *= var_1
            weight[:, :, 2, :, :] *= var_2
            weight[:, :, 3, :, :] *= var_3
            weight[:, :, 4, :, :] *= var_4
            
            filtered_grad = filtered_grad * weight
    
    else:
        raise ValueError(f"Unknown filter_type: {filter_type}")
    
    return filtered_grad

print("✓ Gradient filtering function defined")

In [ ]:
def create_gaussian_kernel(kernel_size=5, sigma=1.0, channels=1, device='cuda:0'):
    """Create a 2D Gaussian kernel for spatial smoothing.
    
    Args:
        kernel_size: Size of the kernel (must be odd)
        sigma: Standard deviation of the Gaussian
        channels: Number of channels (for depthwise convolution)
        device: Device to create the kernel on
    
    Returns:
        kernel: Gaussian kernel tensor [channels, 1, kernel_size, kernel_size]
    """
    # Create 1D Gaussian kernel
    x = torch.arange(kernel_size, dtype=torch.float32, device=device)
    x = x - kernel_size // 2
    gauss_1d = torch.exp(-x**2 / (2 * sigma**2))
    gauss_1d = gauss_1d / gauss_1d.sum()
    
    # Create 2D kernel by outer product
    gauss_2d = gauss_1d.unsqueeze(0) * gauss_1d.unsqueeze(1)
    
    # Expand for depthwise convolution: [channels, 1, H, W]
    kernel = gauss_2d.unsqueeze(0).unsqueeze(0).expand(channels, 1, kernel_size, kernel_size)
    
    return kernel

def apply_spatial_smoothing(tensor, kernel_size=5, sigma=1.0, ocean_mask=None):
    """Apply spatial smoothing filter to tensor with shape (batch, ch, lat, lon).
    
    Uses Gaussian blur for smooth, differentiable filtering. Applies padding to maintain
    spatial dimensions and respects ocean mask if provided.
    
    Args:
        tensor: Input tensor [batch, ch, lat, lon]
        kernel_size: Size of Gaussian kernel (must be odd)
        sigma: Standard deviation of Gaussian (larger = more smoothing)
        ocean_mask: Optional ocean mask [ch, lat, lon] to preserve land/ocean boundary
    
    Returns:
        smoothed: Spatially smoothed tensor [batch, ch, lat, lon]
    """
    batch, channels, height, width = tensor.shape
    device = tensor.device
    
    # Create Gaussian kernel
    kernel = create_gaussian_kernel(kernel_size, sigma, channels, device)
    
    # Apply depthwise convolution (groups=channels for per-channel smoothing)
    padding = kernel_size // 2
    smoothed = nn.functional.conv2d(
        tensor, 
        kernel, 
        padding=padding, 
        groups=channels
    )
    
    # Re-apply ocean mask if provided
    if ocean_mask is not None:
        smoothed = smoothed * ocean_mask.unsqueeze(0)
    
    return smoothed

print("✓ Spatial smoothing function defined")

In [ ]:
def compute_rmse_metrics(predictions, targets, x0_current, x0_reference):
    """Compute RMSE metrics for predictions and initial conditions.
    
    Args:
        predictions: List of prediction tensors, one for each forecast day
        targets: List of target tensors, one for each forecast day
        x0_current: Current initial condition
        x0_reference: Reference initial condition
    
    Returns:
        metrics: Dictionary with RMSE metrics for predictions (averaged over forecast days) and ICs
    """
    with torch.no_grad():
        # Compute prediction RMSE averaged over all forecast days
        rmse_per_channel_sum = None
        
        for y_hat, target_tensor in zip(predictions, targets):
            squared_errors = (target_tensor - y_hat) ** 2
            squared_errors_masked = squared_errors * ocean_mask_all.unsqueeze(0)
            n_ocean_points = ocean_mask_all.unsqueeze(0).sum(dim=(2, 3))
            mse_per_channel = squared_errors_masked.sum(dim=(2, 3)) / (n_ocean_points + 1e-10)
            rmse_per_channel = torch.sqrt(mse_per_channel)
            
            if rmse_per_channel_sum is None:
                rmse_per_channel_sum = rmse_per_channel
            else:
                rmse_per_channel_sum += rmse_per_channel
        
        # Average RMSE over forecast days
        rmse_per_channel = (rmse_per_channel_sum / len(predictions)).cpu().numpy().squeeze(0)
        
        # IC errors
        ic_squared_errors = (x0_reference - x0_current) ** 2
        ic_squared_errors_masked = ic_squared_errors * ocean_mask_all.unsqueeze(0).unsqueeze(0)
        ic_mse_per_channel = ic_squared_errors_masked.sum(dim=(3, 4)) / (n_ocean_points + 1e-10)
        ic_rmse_per_channel = torch.sqrt(ic_mse_per_channel).cpu().numpy().squeeze(0)
        
        # Compute per-variable metrics
        metrics = {
            'pred': {
                'SSH': float(rmse_per_channel[0]),
                'thetao': [float(rmse_per_channel[1:2].mean()), 
                               float(rmse_per_channel[5:15].mean()), 
                               float(rmse_per_channel[45:55].mean())],
                'so': [float(rmse_per_channel[2:3].mean()), 
                           float(rmse_per_channel[15:25].mean()), 
                           float(rmse_per_channel[55:65].mean())],
                'uo': [float(rmse_per_channel[3:4].mean()), 
                           float(rmse_per_channel[25:35].mean()), 
                           float(rmse_per_channel[65:75].mean())],
                'vo': [float(rmse_per_channel[4:5].mean()), 
                           float(rmse_per_channel[35:45].mean()), 
                           float(rmse_per_channel[75:85].mean())],
            },
            'ic': {
                'SSH': float(ic_rmse_per_channel[1, 0]),
                'thetao': [float(ic_rmse_per_channel[1, 1:2].mean()), 
                              float(ic_rmse_per_channel[1, 5:15].mean()), 
                              float(ic_rmse_per_channel[1, 45:55].mean())],
                'so': [float(ic_rmse_per_channel[1, 2:3].mean()), 
                           float(ic_rmse_per_channel[1, 15:25].mean()), 
                           float(ic_rmse_per_channel[1, 55:65].mean())],
                'uo': [float(ic_rmse_per_channel[1, 3:4].mean()), 
                           float(ic_rmse_per_channel[1, 25:35].mean()), 
                           float(ic_rmse_per_channel[1, 65:75].mean())],
                'vo': [float(ic_rmse_per_channel[1, 4:5].mean()), 
                           float(ic_rmse_per_channel[1, 35:45].mean()), 
                           float(ic_rmse_per_channel[1, 75:85].mean())],
            }
        }
        
        return metrics

print("✓ RMSE computation function defined") 

## 6. Visualization Functions

In [ ]:
def plot_rmse_evolution(history, iteration):
    """Plot RMSE evolution over iterations with separate subplots for each depth level."""
    # Create figure with 3 columns (Prediction RMSE, IC RMSE, Loss) and multiple rows
    fig, axes = plt.subplots(14, 3, figsize=(24, 36))
    fig.suptitle(f'RMSE and Loss Evolution - Iteration {iteration}', fontsize=16, fontweight='bold')
    
    variables = ['SSH', 'thetao', 'so', 'uo', 'vo']
    var_titles = ['SSH (Sea Surface Height)', 'Temperature (thetao)', 'Salinity (so)', 
                  'Eastward Velocity (uo)', 'Northward Velocity (vo)']
    loss_keys = ['loss_ssh', 'loss_t', 'loss_s', 'loss_u', 'loss_v']
    depths = ['surface', 'shallow', 'deep']
    
    row_idx = 0
    
    for var, title, loss_key in zip(variables, var_titles, loss_keys):
        pred_rmse = [h['pred'][var] for h in history]
        ic_rmse = [h['ic'][var] for h in history]
        var_loss = [h[loss_key] for h in history]
        
        if isinstance(pred_rmse[0], list):
            # Variable with depth levels
            for depth_idx, depth in enumerate(depths):
                pred_rmse_depth = [p[depth_idx] for p in pred_rmse]
                ic_rmse_depth = [ic[depth_idx] for ic in ic_rmse]
                
                # Prediction RMSE subplot
                axes[row_idx, 0].plot(pred_rmse_depth, 'b-', linewidth=2)
                axes[row_idx, 0].set_xlabel('Iteration', fontsize=10)
                axes[row_idx, 0].set_ylabel('RMSE', fontsize=10)
                axes[row_idx, 0].set_title(f'{title} - {depth} (Pred)', fontsize=11, fontweight='bold')
                axes[row_idx, 0].grid(True, alpha=0.3)
                
                # IC RMSE subplot
                axes[row_idx, 1].plot(ic_rmse_depth, 'r-', linewidth=2)
                axes[row_idx, 1].set_xlabel('Iteration', fontsize=10)
                axes[row_idx, 1].set_ylabel('RMSE', fontsize=10)
                axes[row_idx, 1].set_title(f'{title} - {depth} (IC)', fontsize=11, fontweight='bold')
                axes[row_idx, 1].grid(True, alpha=0.3)
                
                # Loss subplot (only for first depth level of each variable)
                if depth_idx == 0:
                    axes[row_idx, 2].plot(var_loss, 'g-', linewidth=2)
                    axes[row_idx, 2].set_xlabel('Iteration', fontsize=10)
                    axes[row_idx, 2].set_ylabel('Loss', fontsize=10)
                    axes[row_idx, 2].set_title(f'{title} Loss', fontsize=11, fontweight='bold')
                    axes[row_idx, 2].grid(True, alpha=0.3)
                else:
                    # Hide loss subplot for non-first depth levels
                    axes[row_idx, 2].axis('off')
                
                row_idx += 1
        else:
            # SSH (no depth levels)
            # Prediction RMSE subplot
            axes[row_idx, 0].plot(pred_rmse, 'b-', linewidth=2)
            axes[row_idx, 0].set_xlabel('Iteration', fontsize=10)
            axes[row_idx, 0].set_ylabel('RMSE', fontsize=10)
            axes[row_idx, 0].set_title(f'{title} (Pred)', fontsize=11, fontweight='bold')
            axes[row_idx, 0].grid(True, alpha=0.3)
            
            # IC RMSE subplot
            axes[row_idx, 1].plot(ic_rmse, 'r-', linewidth=2)
            axes[row_idx, 1].set_xlabel('Iteration', fontsize=10)
            axes[row_idx, 1].set_ylabel('RMSE', fontsize=10)
            axes[row_idx, 1].set_title(f'{title} (IC)', fontsize=11, fontweight='bold')
            axes[row_idx, 1].grid(True, alpha=0.3)
            
            # Loss subplot
            axes[row_idx, 2].plot(var_loss, 'g-', linewidth=2)
            axes[row_idx, 2].set_xlabel('Iteration', fontsize=10)
            axes[row_idx, 2].set_ylabel('Loss', fontsize=10)
            axes[row_idx, 2].set_title(f'{title} Loss', fontsize=11, fontweight='bold')
            axes[row_idx, 2].grid(True, alpha=0.3)
            
            row_idx += 1
    
    # Total Loss plot (spanning all columns in the last row)
    fig.delaxes(axes[row_idx, 1])
    fig.delaxes(axes[row_idx, 2])
    axes[row_idx, 0].set_position([0.125, 0.025, 0.775, 0.055])
    losses = [h['loss'] for h in history]
    axes[row_idx, 0].plot(losses, 'purple', linewidth=2.5)
    axes[row_idx, 0].set_xlabel('Iteration', fontsize=11)
    axes[row_idx, 0].set_ylabel('Total Loss', fontsize=11)
    axes[row_idx, 0].set_title('Total Loss (Sum of All Variables)', fontsize=12, fontweight='bold')
    axes[row_idx, 0].grid(True, alpha=0.3)
    
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    return fig

print("✓ RMSE evolution plot function defined")

In [ ]:
def plot_predictions(y_hat, y_hat_init, x0_current, x0_reference, coords, iteration, channel_idx=0, time_idx=1):
    """Plot initial vs optimized conditions and predictions in a 2x3 grid.
    
    Args:
        y_hat: Current prediction tensor [1, 85, H, W]
        y_hat_init: Initial prediction tensor [1, 85, H, W]
        x0_current: Current initial condition [1, T, 85, H, W]
        x0_reference: Reference initial condition [1, T, 85, H, W]
        coords: Dictionary with lat/lon coordinates
        iteration: Current iteration number
        channel_idx: Channel index to visualize
        time_idx: Time step index for initial conditions (default=1, last time step)
    """
    with torch.no_grad():
        # Predictions
        pred_opt = y_hat[0, channel_idx].cpu().numpy()
        pred_init = y_hat_init[0, channel_idx].cpu().numpy()
        pred_diff = pred_opt - pred_init
        
        # Initial conditions
        ic_opt = x0_current[0, time_idx, channel_idx].cpu().numpy()
        ic_init = x0_reference[0, time_idx, channel_idx].cpu().numpy()
        ic_diff = ic_opt - ic_init
    
    # Replace zeros with NaN for ocean-only visualization
    # pred_opt = np.where(pred_opt == 0, np.nan, pred_opt)
    # pred_init = np.where(pred_init == 0, np.nan, pred_init)
    # pred_diff = np.where(np.isnan(pred_opt) | np.isnan(pred_init), np.nan, pred_diff)
    
    # ic_opt = np.where(ic_opt == 0, np.nan, ic_opt)
    # ic_init = np.where(ic_init == 0, np.nan, ic_init)
    # ic_diff = np.where(np.isnan(ic_opt) | np.isnan(ic_init), np.nan, ic_diff)
    
    land_nan = np.where(ocean_mask_all[channel_idx].cpu().numpy() == 0, np.nan, 1)
    pred_opt = pred_opt * land_nan
    pred_init = pred_init * land_nan
    pred_diff = pred_diff * land_nan
    
    ic_opt = ic_opt * land_nan
    ic_init = ic_init * land_nan
    ic_diff = ic_diff * land_nan
    
    # Get lat/lon coordinates
    lons = coords['lon']
    lats = coords['lat']
    
    fig, axes = plt.subplots(2, 3, figsize=(22, 12))
    fig.suptitle(f'Channel {channel_idx} - Iteration {iteration}', fontsize=16, fontweight='bold')
    
    # Row 1: Initial Conditions
    # Initial IC
    im00 = axes[0, 0].pcolormesh(lons, lats, ic_init, cmap='viridis', shading='auto')
    axes[0, 0].set_title('Initial Condition (Reference)', fontsize=12)
    axes[0, 0].set_xlabel('Longitude (°E)', fontsize=11)
    axes[0, 0].set_ylabel('Latitude (°N)', fontsize=11)
    axes[0, 0].set_aspect('equal')
    plt.colorbar(im00, ax=axes[0, 0], label='Value', fraction=0.025, pad=0.04)
    
    # Optimized IC
    im01 = axes[0, 1].pcolormesh(lons, lats, ic_opt, cmap='viridis', shading='auto')
    axes[0, 1].set_title('Optimized Initial Condition', fontsize=12)
    axes[0, 1].set_xlabel('Longitude (°E)', fontsize=11)
    axes[0, 1].set_ylabel('Latitude (°N)', fontsize=11)
    axes[0, 1].set_aspect('equal')
    plt.colorbar(im01, ax=axes[0, 1], label='Value', fraction=0.025, pad=0.04)
    
    # IC Difference
    max_abs_ic_diff = np.nanmax(np.abs(ic_diff))
    im02 = axes[0, 2].pcolormesh(lons, lats, ic_diff, cmap='RdBu_r', shading='auto',
                                 vmin=-max_abs_ic_diff, vmax=max_abs_ic_diff)
    axes[0, 2].set_title('IC Difference (Opt - Init)', fontsize=12)
    axes[0, 2].set_xlabel('Longitude (°E)', fontsize=11)
    axes[0, 2].set_ylabel('Latitude (°N)', fontsize=11)
    axes[0, 2].set_aspect('equal')
    plt.colorbar(im02, ax=axes[0, 2], label='Difference', fraction=0.025, pad=0.04)
    
    # Row 2: Predictions
    # Initial Prediction
    im10 = axes[1, 0].pcolormesh(lons, lats, pred_init, cmap='viridis', shading='auto')
    axes[1, 0].set_title('Initial Prediction', fontsize=12)
    axes[1, 0].set_xlabel('Longitude (°E)', fontsize=11)
    axes[1, 0].set_ylabel('Latitude (°N)', fontsize=11)
    axes[1, 0].set_aspect('equal')
    plt.colorbar(im10, ax=axes[1, 0], label='Value', fraction=0.025, pad=0.04)
    
    # Optimized Prediction
    im11 = axes[1, 1].pcolormesh(lons, lats, pred_opt, cmap='viridis', shading='auto')
    axes[1, 1].set_title('Optimized Prediction', fontsize=12)
    axes[1, 1].set_xlabel('Longitude (°E)', fontsize=11)
    axes[1, 1].set_ylabel('Latitude (°N)', fontsize=11)
    axes[1, 1].set_aspect('equal')
    plt.colorbar(im11, ax=axes[1, 1], label='Value', fraction=0.025, pad=0.04)
    
    # Prediction Difference
    max_abs_pred_diff = np.nanmax(np.abs(pred_diff))
    im12 = axes[1, 2].pcolormesh(lons, lats, pred_diff, cmap='RdBu_r', shading='auto',
                                 vmin=-max_abs_pred_diff, vmax=max_abs_pred_diff)
    axes[1, 2].set_title('Prediction Difference (Opt - Init)', fontsize=12)
    axes[1, 2].set_xlabel('Longitude (°E)', fontsize=11)
    axes[1, 2].set_ylabel('Latitude (°N)', fontsize=11)
    axes[1, 2].set_aspect('equal')
    plt.colorbar(im12, ax=axes[1, 2], label='Difference', fraction=0.025, pad=0.04)
    
    plt.tight_layout()
    return fig

print("✓ Prediction visualization function defined")

In [ ]:
def plot_gradient(gradient : torch.tensor, 
                  iteration : int,
                  ch : int, 
                  coords : dict) -> None :

    # Visualize gradients for channel at both time steps
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle(f'Standardized Gradients (Channel {ch}) - Iteration {iteration + 1}', fontsize=16, fontweight='bold')
    
    for t_idx in range(2):
        grad_t_ch = gradient[0, t_idx, ch].cpu().numpy()
        land_nan = np.where(ocean_mask_all[ch].cpu().numpy() == 0, np.nan, 1) # Already masked grad
        grad_t_ch = grad_t_ch * land_nan
        max_abs_grad = np.nanmax(np.abs(grad_t_ch))
        im = axes[t_idx].pcolormesh(coords['lon'], coords['lat'], grad_t_ch, 
                                            cmap='RdBu_r', shading='auto',
                                            vmin=-max_abs_grad, vmax=max_abs_grad)
        axes[t_idx].set_title(f'Time Step {t_idx}', fontsize=12)
        axes[t_idx].set_xlabel('Longitude (°E)', fontsize=11)
        axes[t_idx].set_ylabel('Latitude (°N)', fontsize=11)
        axes[t_idx].set_aspect('equal')
        plt.colorbar(im, ax=axes[t_idx], label='Standardized Gradient', fraction=0.025, pad=0.04)
    
    plt.tight_layout()
    return fig

In [ ]:
def plot_grad_histogram(gradient : torch.tensor, 
                        iteration : int) -> None :
    "Plot normalized histogram of gradients with statistics and log scale."
    
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    fig.suptitle(f'Standardized Gradient Histograms - Iteration {iteration}', fontsize=16, fontweight='bold')
    
    var_names = ['SSH', 'THETAO', 'SO', 'UO', 'VO']
    colors = ['red', 'blue', 'green', 'orange', 'purple']
    
    for t_idx in range(2):
        # Collect all gradients for this timestep
        all_grads = {}
        stats_text = f"Timestep t={t_idx}\n" + "="*40 + "\n"
        
        for ch_idx, var_name in enumerate(var_names):
            if ch_idx == 0:  # SSH (2D)
                grad_data = gradient[0, t_idx, ch_idx].cpu().numpy()
            else:  # 3D variables
                grad_data = gradient[0, t_idx, ch_idx, :, :].cpu().numpy()
            
            # Flatten and remove NaN values (land points)
            grad_flat = grad_data.flatten()
            grad_flat = grad_flat[~np.isnan(grad_flat)]
            all_grads[var_name] = grad_flat
            
            # Statistics
            n_total = len(grad_flat)
            n_zero = np.sum(np.abs(grad_flat) < 1e-10)
            n_nonzero = n_total - n_zero
            pct_zero = (n_zero / n_total) * 100
            mean_val = np.mean(grad_flat)
            std_val = np.std(grad_flat)
            min_val = np.min(grad_flat)
            max_val = np.max(grad_flat)
            
            stats_text += f"{var_name:8s}: zero={pct_zero:5.1f}% | mean={mean_val:.2e} | std={std_val:.2e}\n"
            stats_text += f"          min={min_val:.2e} | max={max_val:.2e}\n"
        
        print(stats_text)
        
        # Plot 1: Linear scale (full range)
        ax_linear = axes[t_idx, 0]
        for ch_idx, var_name in enumerate(var_names):
            grad_flat = all_grads[var_name]
            # Compute histogram counts
            hist_vals, bin_edges = np.histogram(grad_flat, bins=100)
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            ax_linear.plot(bin_centers, hist_vals, color=colors[ch_idx], 
                          linewidth=2.5, label=var_name, alpha=0.9)
            mean_val = np.mean(grad_flat)
            ax_linear.axvline(mean_val, color=colors[ch_idx], linestyle='--', 
                             linewidth=1.5, alpha=0.7)
        
        ax_linear.set_xlabel('Standardized Gradient Value', fontsize=11)
        ax_linear.set_ylabel('Count', fontsize=11)
        ax_linear.set_title(f'Timestep t={t_idx} - Linear Scale', fontsize=12)
        ax_linear.grid(True, alpha=0.3)
        ax_linear.legend(fontsize=9, loc='best')
        
        # Plot 2: Log scale
        ax_log = axes[t_idx, 1]
        for ch_idx, var_name in enumerate(var_names):
            grad_flat = all_grads[var_name]
            # Remove exact zeros for log scale
            grad_nonzero = grad_flat[np.abs(grad_flat) > 1e-15]
            if len(grad_nonzero) > 0:
                # Compute histogram counts
                hist_vals, bin_edges = np.histogram(grad_nonzero, bins=100)
                bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
                ax_log.plot(bin_centers, hist_vals, color=colors[ch_idx], 
                           linewidth=2.5, label=var_name, alpha=0.9)
                mean_val = np.mean(grad_nonzero)
                ax_log.axvline(mean_val, color=colors[ch_idx], linestyle='--', 
                              linewidth=1.5, alpha=0.7)
        
        ax_log.set_xlabel('Standardized Gradient Value', fontsize=11)
        ax_log.set_ylabel('Count (log scale)', fontsize=11)
        ax_log.set_title(f'Timestep t={t_idx} - Log Scale (non-zero only)', fontsize=12)
        ax_log.set_yscale('log')
        ax_log.grid(True, alpha=0.3)
        ax_log.legend(fontsize=9, loc='best')
    
    plt.tight_layout()
    return fig

## 7. Initial State Analysis

In [ ]:
# Compute initial predictions
print("Computing initial predictions...")
with torch.no_grad():
    x0_in_std_init, predictions_init = forward(x0)

    initial_metrics = compute_rmse_metrics(predictions_init, target_tensors, x0, x0_ref)
    total_loss_init, total_nloss_init = compute_loss(
        predictions_init, 
        target_tensors,
        apply_smoothing_filter=CONFIG['spatial_smoothing'],
        smooth_kernel_size=CONFIG['smooth_kernel_size'],
        smooth_sigma=CONFIG['smooth_sigma']
    )
    
    nloss_input1_init = total_nloss_init[:, 0:5]
    nloss_input2_init = total_nloss_init[:, 5:45]
    nloss_input3_init = total_nloss_init[:, 45:85]
    
    nloss_ssh_init = total_nloss_init[:, 0:1]
    nloss_t_init = torch.cat([total_nloss_init[:, 1:2], total_nloss_init[:, 5:15], total_nloss_init[:, 45:55]], dim=1)
    nloss_s_init = torch.cat([total_nloss_init[:, 2:3], total_nloss_init[:, 15:25], total_nloss_init[:, 55:65]], dim=1)
    nloss_u_init = torch.cat([total_nloss_init[:, 3:4], total_nloss_init[:, 25:35], total_nloss_init[:, 65:75]], dim=1)
    nloss_v_init = torch.cat([total_nloss_init[:, 4:5], total_nloss_init[:, 35:45], total_nloss_init[:, 75:85]], dim=1)
    
print("\n" + "="*60)
print("INITIAL STATE ANALYSIS")
print("="*60)
print(f"Forecast Horizon: {CONFIG['forecast_horizon']} days")
print(f"Total Loss (summed over {CONFIG['forecast_horizon']} days): {total_nloss_init.sum().item():.6f}")
print(f"\nPrediction RMSE (averaged over {CONFIG['forecast_horizon']} days):")
for var, rmse in initial_metrics['pred'].items():
    if isinstance(rmse, list):
        rmse_str = [f"{r:.6e}" for r in rmse]
        print(f"  {var:10s}: {rmse_str}")
    else:
        print(f"  {var:10s}: {rmse:.6e}")
print(f"\nInitial Condition RMSE:")
for var, rmse in initial_metrics['ic'].items():
    if isinstance(rmse, list):
        rmse_str = [f"{r:.6e}" for r in rmse]
        print(f"  {var:10s}: {rmse_str}")
    else:
        print(f"  {var:10s}: {rmse:.6e}")
print("="*60)

In [ ]:
# Visualize initial predictions (SSH channel)
channel_to_plot = 0  # SSH channel
# Use the last forecast day's prediction for visualization
fig = plot_predictions(predictions_init[-1], predictions_init[-1], x0, x0_ref, coords, 0, channel_idx=channel_to_plot)
# plt.savefig(f"{CONFIG['output_dir']}/initial_prediction_ch{channel_to_plot}.png", dpi=150, bbox_inches='tight')
plt.show()

## 8. Optimization Loop

**Instructions:** Run this cell to start the optimization. 
It will automatically display visualizations every N iterations (configured in `plot_frequency`). 
You can stop it at any time using the stop button.

In [ ]:
# Initialize tracking variables
history = []
best_loss = float('inf')
best_predictions = None  # Store list of predictions for all forecast days
best_x0 = None
best_iteration = 0

# Store initial metrics
history.append({
    'loss': total_nloss_init.sum().item(),
    'loss_input1': nloss_input1_init.sum().item(),
    'loss_input2': nloss_input2_init.sum().item(),
    'loss_input3': nloss_input3_init.sum().item(),
    'loss_ssh': nloss_ssh_init.sum().item(),
    'loss_t': nloss_t_init.sum().item(),
    'loss_s': nloss_s_init.sum().item(),
    'loss_u': nloss_u_init.sum().item(),
    'loss_v': nloss_v_init.sum().item(),
    'pred': initial_metrics['pred'],
    'ic': initial_metrics['ic']
})

print("\n" + "="*60)
print("STARTING OPTIMIZATION")
print("="*60)
print(f"You can stop at any time and inspect the results")
print(f"Plots will be displayed every {CONFIG['plot_frequency']} iterations")
print("="*60)

In [ ]:
for iteration in range(CONFIG['num_iterations']):
    # Zero gradients
    if x0.grad is not None:
        x0.grad.zero_()
    
    # Forward pass - returns list of predictions
    x0_in_std, predictions = forward(x0)
    
    # Compute loss - sum over all forecast days
    total_loss, total_nloss = compute_loss(
        predictions, 
        target_tensors,
        apply_smoothing_filter=CONFIG['spatial_smoothing'],
        smooth_kernel_size=CONFIG['smooth_kernel_size'],
        smooth_sigma=CONFIG['smooth_sigma']
    )
    
    nloss_input1 = total_nloss[:, 0:5]
    nloss_input2 = total_nloss[:, 5:45]
    nloss_input3 = total_nloss[:, 45:85]
    
    nloss_ssh = total_nloss[:, 0:1]
    nloss_t = torch.cat([total_nloss[:, 1:2], total_nloss[:, 5:15], total_nloss[:, 45:55]], dim=1)
    nloss_s = torch.cat([total_nloss[:, 2:3], total_nloss[:, 15:25], total_nloss[:, 55:65]], dim=1)
    nloss_u = torch.cat([total_nloss[:, 3:4], total_nloss[:, 25:35], total_nloss[:, 65:75]], dim=1)
    nloss_v = torch.cat([total_nloss[:, 4:5], total_nloss[:, 35:45], total_nloss[:, 75:85]], dim=1)
    
    # Backward pass
    grads = torch.autograd.grad(nloss_input1.sum(), x0, create_graph=False)
    
    # Apply gradient filter
    filtered_grad = apply_gradient_filter(grads[0], x0, filter_type=CONFIG['gradient_filter'])
    
    # Apply ocean mask to gradients and update
    with torch.no_grad():
        masked_grad = filtered_grad * ocean_mask_all.unsqueeze(0).unsqueeze(0)
        
        # Gradient descent update
        x0 = x0 - CONFIG['learning_rate'] * masked_grad
        
        x0.requires_grad = True
    
    # Compute metrics
    with torch.no_grad() :
        metrics = compute_rmse_metrics(predictions, target_tensors, x0, x0_ref)
    
    # Store history
    history.append({
        'loss': total_nloss.sum().item(),
        'loss_input1': nloss_input1.sum().item(),
        'loss_input2': nloss_input2.sum().item(),
        'loss_input3': nloss_input3.sum().item(),
        'loss_ssh': nloss_ssh.sum().item(),
        'loss_t': nloss_t.sum().item(),
        'loss_s': nloss_s.sum().item(),
        'loss_u': nloss_u.sum().item(),
        'loss_v': nloss_v.sum().item(),
        'pred': metrics['pred'],
        'ic': metrics['ic']
    })
    
    # Update best - store last prediction in forecast horizon
    if total_nloss.sum().item() < best_loss:
        best_loss = total_nloss.sum().item()
        best_predictions = [pred.detach().clone() for pred in predictions]
        best_x0 = x0.detach().clone()
        best_iteration = iteration + 1
    
    # Print and visualize
    if (iteration + 1) % CONFIG['plot_frequency'] == 0 or iteration == 0:
        clear_output(wait=True)
        
        print(f"\n{'='*60}")
        print(f"Iteration {iteration + 1}/{CONFIG['num_iterations']}")
        print(f"Gradient Filter: {CONFIG['gradient_filter']}")
        print(f"Forecast Horizon: {CONFIG['forecast_horizon']} days")
        print(f"{'='*60}")
        print(f"Total Loss (summed over {CONFIG['forecast_horizon']} days): {total_nloss.sum().item():.6f}")
        print(f"Best Loss: {best_loss:.6f} (iteration {best_iteration})")
        print(f"\nPrediction RMSE (averaged over {CONFIG['forecast_horizon']} days):")
        for var, rmse in metrics['pred'].items():
            if isinstance(rmse, list):
                rmse_str = [f"{r:.6e}" for r in rmse]
                print(f"  {var:10s}: {rmse_str}")
            else:
                print(f"  {var:10s}: {rmse:.6e}")
        print(f"\nInitial Condition RMSE:")
        for var, rmse in metrics['ic'].items():
            if isinstance(rmse, list):
                rmse_str = [f"{r:.6e}" for r in rmse]
                print(f"  {var:10s}: {rmse_str}")
            else:
                print(f"  {var:10s}: {rmse:.6e}")
        
        # Standardize gradient for visualization (per channel, per timestep)
        standardized_grad = masked_grad.clone()
        for t_idx in range(standardized_grad.shape[1]):
            for ch_idx in range(standardized_grad.shape[2]):
                grad_ch = standardized_grad[0, t_idx, ch_idx]
                # Get ocean points only (non-zero in mask)
                ocean_points = grad_ch[ocean_mask_all[ch_idx] > 0]
                if len(ocean_points) > 0:
                    mean_val = ocean_points.mean()
                    std_val = ocean_points.std()
                    if std_val > 1e-10:
                        # Standardize: (x - mean) / std
                        standardized_grad[0, t_idx, ch_idx] = (grad_ch - mean_val) / std_val
                        # Re-apply mask to ensure land points are zero
                        standardized_grad[0, t_idx, ch_idx] *= ocean_mask_all[ch_idx]
        
        print(f"\nGradient Statistics (after {CONFIG['gradient_filter']} filter):")
        for ch_idx, var_name in enumerate(['SSH', 'THETAO', 'SO', 'UO', 'VO']):
            grad_ch = masked_grad[0, 1, ch_idx]  # Use last timestep
            ocean_points = grad_ch[ocean_mask_all[ch_idx] > 0]
            if len(ocean_points) > 0:
                print(f"  {var_name:8s}: mean={ocean_points.mean():.2e}, std={ocean_points.std():.2e}, "
                      f"min={ocean_points.min():.2e}, max={ocean_points.max():.2e}")
        
        # Plot gradient for SST (channel 1)
        # No, plot every variable gradient
        for i in range(5) :
            fig_grad = plot_gradient(standardized_grad, iteration + 1, ch=i, coords=coords)
        # plt.savefig(f"{CONFIG['output_dir']}/gradient_iter{iteration+1:04d}.png", 
        #            dpi=150, bbox_inches='tight')
        plt.show()
        
        fig_histo = plot_grad_histogram(standardized_grad, iteration + 1)
        plt.show()
        
        # Plot RMSE evolution
        fig_rmse = plot_rmse_evolution(history, iteration + 1)
        # plt.savefig(f"{CONFIG['output_dir']}/rmse_evolution_iter{iteration+1:04d}.png", 
        #            dpi=150, bbox_inches='tight')
        plt.show()
        
        # Plot predictions - use last forecast day
        fig_pred = plot_predictions(predictions[-1], predictions_init[-1], x0, x0_ref, coords, iteration + 1, channel_idx=1)
        # plt.savefig(f"{CONFIG['output_dir']}/predictions_iter{iteration+1:04d}.png", 
        #            dpi=150, bbox_inches='tight')
        plt.show()
    
    # Memory cleanup
    del predictions, total_nloss, nloss_ssh, nloss_t, nloss_s, nloss_u, nloss_v , grads
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("OPTIMIZATION COMPLETE")
print("="*60)

In [ ]:
# Plot final RMSE evolution
fig = plot_rmse_evolution(history, len(history)-1)
plt.savefig(f"{CONFIG['output_dir']}/rmse_evolution_final.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot best predictions for multiple channels
channels_to_plot = [0, 1, 2, 3, 4]  # SSH, thetao, so, uo, vo (surface)
channel_names = ['SSH', 'thetao_surface', 'so_surface', 'uo_surface', 'vo_surface']

for ch, name in zip(channels_to_plot, channel_names):
    print(f"\nPlotting channel {ch}: {name}")
    # Use last forecast day's prediction for visualization
    fig = plot_predictions(best_predictions[-1], predictions_init[-1], best_x0, x0_ref, coords, best_iteration, channel_idx=ch)
    plt.savefig(f"{CONFIG['output_dir']}/best_prediction_ch{ch}_{name}.png", 
               dpi=150, bbox_inches='tight')
    plt.show()

## 10. Save Results

In [ ]:
# Save history as DataFrame and JSON
import json

# Save history
history_df = pd.DataFrame([
    {
        'iteration': i,
        'loss': h['loss'],
        **{f'pred_{k}': v for k, v in h['pred'].items()},
        **{f'ic_{k}': v for k, v in h['ic'].items()}
    }
    for i, h in enumerate(history)
])

history_df.to_csv(f"{CONFIG['output_dir']}/optimization_history.csv", index=False)
print(f"✓ Saved optimization history to {CONFIG['output_dir']}/optimization_history.csv")

# Display first few rows
display(history_df.head(10))

In [ ]:
# Save configuration and results
results = {
    'config': CONFIG,
    'gradient_filter': CONFIG['gradient_filter'],
    'forecast_horizon': CONFIG['forecast_horizon'],
    'initial_loss': history[0]['loss'],
    'final_loss': history[-1]['loss'],
    'best_loss': best_loss,
    'best_iteration': best_iteration,
    'initial_metrics': history[0],
    'final_metrics': history[-1],
}

with open(f"{CONFIG['output_dir']}/results.json", 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"✓ Saved results to {CONFIG['output_dir']}/results.json")

In [ ]:
# Save optimized initial conditions
with torch.no_grad():
    opt_x0_np = best_x0.cpu().numpy().squeeze(0)  # [T, 85, H, W]

ds_opt = xr.Dataset({
    'data': (['time', 'ch', 'lat', 'lon'], opt_x0_np)
}, coords={
    'time': coords['time'],
    'ch': np.arange(85),
    'lat': coords['lat'],
    'lon': coords['lon']
})

ds_opt.attrs['description'] = 'Optimized initial conditions from gradient descent'
ds_opt.attrs['creation_date'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
ds_opt.attrs['best_loss'] = float(best_loss)
ds_opt.attrs['best_iteration'] = best_iteration
ds_opt.attrs['learning_rate'] = CONFIG['learning_rate']
ds_opt.attrs['num_iterations'] = CONFIG['num_iterations']

encoding = {'data': {'zlib': True, 'complevel': 4}}
ds_opt.to_netcdf(f"{CONFIG['output_dir']}/optimized_initial_conditions.nc", encoding=encoding)
print(f"✓ Saved optimized initial conditions to {CONFIG['output_dir']}/optimized_initial_conditions.nc")

In [ ]:
# Save best predictions for all forecast days
with torch.no_grad():
    # Stack all predictions: [forecast_days, 1, 85, H, W] -> [forecast_days, 85, H, W]
    best_preds_stacked = torch.cat([pred for pred in best_predictions], dim=0).cpu().numpy()

ds_pred = xr.Dataset({
    'data': (['time', 'ch', 'lat', 'lon'], best_preds_stacked)
}, coords={
    'time': coords['target_times'],  # All target times
    'ch': np.arange(85),
    'lat': coords['lat'],
    'lon': coords['lon']
})

ds_pred.attrs['description'] = f'Best forecast predictions from optimized initial conditions ({CONFIG["forecast_horizon"]} days)'
ds_pred.attrs['creation_date'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
ds_pred.attrs['best_loss'] = float(best_loss)
ds_pred.attrs['best_iteration'] = best_iteration
ds_pred.attrs['forecast_horizon'] = CONFIG['forecast_horizon']

encoding = {'data': {'zlib': True, 'complevel': 4}}
ds_pred.to_netcdf(f"{CONFIG['output_dir']}/best_predictions.nc", encoding=encoding)
print(f"✓ Saved best predictions ({CONFIG['forecast_horizon']} days) to {CONFIG['output_dir']}/best_predictions.nc")

## 11. Optional: Inspect Specific Iterations

You can manually inspect predictions at any iteration stored in the history.

In [ ]:
# Example: Plot RMSE for a specific variable over time
variable = 'SSH'  # Change to: 'thetao', 'so', 'uo', or 'vo'

iterations = list(range(len(history)))
pred_rmse = [h['pred'][variable] for h in history]
ic_rmse = [h['ic'][variable] for h in history]

plt.figure(figsize=(12, 6))
plt.plot(iterations, pred_rmse, 'b-', linewidth=2, label=f'{variable} Prediction RMSE')
plt.plot(iterations, ic_rmse, 'r--', linewidth=2, label=f'{variable} IC RMSE')
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('RMSE', fontsize=12)
plt.title(f'{variable} RMSE Evolution', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

All results have been saved to: `{CONFIG['output_dir']}`

### Saved files:
- `optimization_history.csv` - Complete history of losses and RMSE values
- `results.json` - Configuration and summary statistics
- `optimized_initial_conditions.nc` - Best optimized initial conditions
- `best_predictions.nc` - Best forecast predictions
- Multiple PNG visualization files

You can now reload these files for further analysis or use the optimized initial conditions for new forecasts.